In [6]:
# Cell 1
# Mục đích: dựng lại kết nối và hàm đo chi phí (notebook mới phải định nghĩa lại).
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")  # kênh kết nối tới BigQuery

def mb(n):
    # Đổi byte sang MB; trả None nếu n là None để lỗi "đọc byte quá sớm" lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def stats(sql, max_mb=100):
    # Chạy thật, tắt cache, đặt trần byte billed làm cầu chì
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,
        maximum_bytes_billed=max_mb * 1024 * 1024,  # đổi MB sang byte
    )
    job = client.query(sql, job_config=cfg)   # gửi job, KHÔNG chờ
    df = job.result().to_dataframe()          # chờ xong rồi đưa kết quả vào df
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df                                 # byte lấy từ job, df chỉ là kết quả

print("Cell 1 OK")

Cell 1 OK


In [13]:
# Cell 2
# Mục đích: lấy số chuẩn từ SQL để đối chiếu với measure Power BI.
# Chi phí dự kiến: đọc 2 bảng, billed khoảng 20 MB.
sql = f"""
SELECT
  p.department,
  ROUND(SUM(f.sale_price), 2) AS rev_all,
  ROUND(SUM(IF(EXTRACT(YEAR FROM f.order_date) = 2025, f.sale_price, 0)), 2) AS rev_2025
FROM `{PROJECT}.dbt_dev.fct_net_sales` AS f
LEFT JOIN `{PROJECT}.dbt_dev.dim_products` AS p
  ON f.product_id = p.product_id
GROUP BY p.department
ORDER BY p.department
"""
df = stats(sql)
print(len(df))   # mong đợi 2 dòng
df

processed: 1.43 MB | billed: 20.0 MB
2


,department,rev_all,rev_2025
0,Men,1418890.23,323181.07
1,Women,1288444.65,292179.14


In [14]:
# Cell 3
# Mục đích: lấy số tổng của bản chụp mới để đối chiếu với Power BI sau Refresh.
# Chi phí dự kiến: đọc 1 bảng, billed khoảng 10 MB.
sql = f"""
SELECT
  COUNT(*) AS so_dong,
  COUNT(DISTINCT order_item_id) AS so_id,
  ROUND(SUM(sale_price), 2) AS net_revenue,
  ROUND(SUM(gross_profit), 2) AS gross_profit,
  MIN(order_date) AS ngay_dau,
  MAX(order_date) AS ngay_cuoi
FROM `{PROJECT}.dbt_dev.fct_net_sales`
"""
df = stats(sql)
print(len(df))   # mong đợi 1 dòng
df

processed: 1.38 MB | billed: 10.0 MB
1


,so_dong,so_id,net_revenue,gross_profit,ngay_dau,ngay_cuoi
0,45285,45285,2707334.88,1405850.55,2019-01-21,2026-10-22
